# Llevando a bronze data histórica y estática

In [0]:
from pyspark.sql.functions import *

## Llevando las compras históricas

In [0]:
dbutils.widgets.text("catalog_name", "finalproject")
dbutils.widgets.text("adls_account_name", "safinalproject")
dbutils.widgets.text("adls_container", "raw-data-prueba")

In [0]:
catalog_name = dbutils.widgets.get('catalog_name')
adls_account_name = dbutils.widgets.get("adls_account_name")
adls_container = dbutils.widgets.get("adls_container")

### Llevando las compras presencial

In [0]:
df_compras_presencial = spark.read.option('header', 'true').option('sep', ';').csv(f'abfss://{adls_container}@{adls_account_name}.dfs.core.windows.net/historico/compras/Presencial.csv')

### Llevando las compras online

In [0]:
df_compras_online = spark.read.options(multiline=True).json(f'abfss://{adls_container}@{adls_account_name}.dfs.core.windows.net/historico/compras/Online.json')

### Uniendo para tener compras históricas

In [0]:
df_compras_presencial = df_compras_presencial.drop('Subtotal').withColumn('tipo_compra', lit('Presencial'))
df_compras_online = (
    df_compras_online
    .withColumnRenamed('ClienteCode', 'Cliente_Code')
    .withColumnRenamed('FechaEntrega', 'Fecha_Entrega')
    .withColumnRenamed('FechaEnvio', 'Fecha_Envio')
    .withColumnRenamed('FechaOrden', 'Fecha_Orden')
    .withColumnRenamed('MetodoPago', 'Metodo_Pago')
    .withColumnRenamed('TipoCliente', 'Tipo_Cliente')
    .withColumn('tipo_compra', lit('Online'))
)
df_compras = df_compras_presencial.union(df_compras_online)

### Normalizando las columnas (colocamos todo en `minúsuclas`)

In [0]:
df_compras = df_compras.toDF(*[col.lower() for col in df_compras.columns])

### Exportamos el dataframe a la capa bronze

In [0]:
df_compras.write.mode('overwrite').format('delta').saveAsTable(f'{catalog_name}.bronze.compras_historicos')

## Llevando los detalles históricos

### Extrayendo los detalles históricos

In [0]:
df_detalles = spark.read.option('header', 'true').option('sep', '|').csv(f'abfss://{adls_container}@{adls_account_name}.dfs.core.windows.net/historico/detalles/*csv')

### Normalizando las columnas (colocamos todo en `minúsuclas`)

In [0]:
df_detalles = df_detalles.toDF(*[col.lower() for col in df_detalles.columns])

### Añadimos la columna tienda

In [0]:
df_detalles_with = df_detalles.withColumn('tienda', split(col('_metadata.file_name'),'\\.')[0])

### Compartiendo los detalles históricos a la capa Bronze

In [0]:
(
  df_detalles_with.write
  .mode('overwrite')
  .option('overwriteSchema',True)
  .format('delta')
  .saveAsTable(f'{catalog_name}.bronze.detalles_historicos')
)